# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pranajit04/Machine-Learning-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content page (client_hash_id + content_hash_id)
on one day, from fact_content_daily_performance.

**Time window:** month=2026-03 (a mid-panel month) — never the final month
(2026-06), since that's the sealed outcome window any label would look into.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub pandas

import os, getpass, duckdb
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Prove the grain: one row really is one client+content+day
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()
print("Rows breaking the 1-row-per-day rule:", len(grain_check))


Paste your Hugging Face READ token (hf_): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows breaking the 1-row-per-day rule: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** imp_month, clk_month, avg_pos, ctr, pos_volatility — engineered
from daily impressions/clicks/position within the month window.

**Label:** is_declining — derived from click-through-rate trend (proxy, not
ground truth).

**Context:** client_hash_id, content_hash_id — used for grouping/joins only,
never as predictive features (they're just IDs).

**Excluded:** any row from month=2026-06 (final month) — excluded because
it's the sealed test window and using it now would let outcome data leak
into development decisions.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show fields actually pulled into each bucket
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_month,
           SUM(gsc_clicks)      AS clk_month,
           AVG(gsc_avg_position) AS avg_pos,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr,
           STDDEV(gsc_avg_position) AS pos_volatility
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY 1,2
""").df()

features_clean = features[features['imp_month'] >= 50].copy()
print("Rows before filter:", len(features), "| after filter:", len(features_clean))
print("CTR median (all rows):", features['ctr'].median())
print("CTR median (filtered):", features_clean['ctr'].median())

features_clean['is_declining'] = (features_clean['ctr'] < features_clean['ctr'].median()).astype(int)
print("\nLabel balance:\n", features_clean['is_declining'].value_counts(normalize=True))
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows before filter: 331437 | after filter: 116114
CTR median (all rows): 0.0
CTR median (filtered): 0.0008576329331046312

Label balance:
 is_declining
0    0.500009
1    0.499991
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,imp_month,clk_month,avg_pos,ctr,pos_volatility
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,0.000000,2.678448
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,0.006645,1.809163
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,0.001235,1.853147
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,0.000000,9.774390
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,0.003229,0.691448


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries: row count + date span, availability filtered
with IS TRUE, and the deliberate leakage trap (added, scored, then removed).
The leak wasn't a perfect duplicate of the label, so the score jump (0.680 →
0.793) is real but moderate — a reminder that leakage doesn't always look
like near-100% accuracy; even a partial leak measurably inflates the score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Row count + date span
span = con.sql(f"""
    SELECT COUNT(*) as n_rows, MIN(report_date) as first_day, MAX(report_date) as last_day
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(span)

# Availability, filtered with IS TRUE
avail = con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_clients']} c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
      AND c.has_gsc_access IS TRUE
""").df()
print(avail)

# --- Filter out near-zero-signal rows before labeling ---
features_clean = features[features['imp_month'] >= 50].copy()
print("Rows before filter:", len(features), "| after filter:", len(features_clean))
print("CTR median (all rows):", features['ctr'].median())
print("CTR median (filtered):", features_clean['ctr'].median())

features_clean['is_declining'] = (features_clean['ctr'] < features_clean['ctr'].median()).astype(int)
print("\nLabel balance:\n", features_clean['is_declining'].value_counts(normalize=True))

# --- THE TRAP: deliberately add a label-derived column ---
features_clean['fake_leak_feature'] = features_clean['clk_month'] / (features_clean['imp_month'] + 1)  # ~= ctr, near-duplicate of the label

from sklearn.linear_model import LogisticRegression
X_leaky = features_clean[['imp_month','fake_leak_feature']].fillna(0)
y = features_clean['is_declining']
leaky_model = LogisticRegression().fit(X_leaky, y)
print("\nLeaky score (inflated, don't trust this):", leaky_model.score(X_leaky, y))

# Now remove the leak and keep the honest number
X_honest = features_clean[['imp_month','avg_pos','pos_volatility']].fillna(0)
honest_model = LogisticRegression().fit(X_honest, y)
print("Honest score (real number):", honest_model.score(X_honest, y))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows  first_day   last_day
0  9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   available_rows
0         9829226
Rows before filter: 331437 | after filter: 116114
CTR median (all rows): 0.0
CTR median (filtered): 0.0008576329331046312

Label balance:
 is_declining
0    0.500009
1    0.499991
Name: proportion, dtype: float64

Leaky score (inflated, don't trust this): 0.7933065780181545
Honest score (real number): 0.6795562981208123


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me *why* a page's performance changed — only *that*
it changed. The panel is also unbalanced: not every client has data starting
this early (see dim_clients.gsc_data_start), so early months under-represent
newer clients. And GSC-only clients (no GA4) miss engagement signals that
GSC+GA4 clients have, which could bias any pattern that relies on engagement.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unbalanced panel + GSC-only gap directly
limits_check = con.sql(f"""
    SELECT access_profile, COUNT(*) as n_clients,
           MIN(gsc_data_start) as earliest_start
    FROM {TABLES['dim_clients']}
    GROUP BY access_profile
""").df()
print(limits_check)


                         access_profile  n_clients earliest_start
0                              gsc_only         14     2025-06-07
1                              ga4_only          1            NaT
2                           gsc_and_ga4         53     2025-01-27
3         no_search_or_analytics_access         26     2025-11-05
4  source_only_missing_client_dimension         10     2025-11-05


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.